# S03 · Atom Mapping as Graph Morphism: MCS, RXNMapper, and ITS Equivalence

<div class="alert alert-block alert-info">
<b>Welcome to S03.</b><br>
This talktorial extends <b>S01–S02</b> from single-molecule matching to the
<b>reaction setting</b>. We move from pattern alignment to
<b>atom mapping</b>, treating mappings as explicit graph morphisms between
reaction sides.
</div>

<div class="alert alert-block alert-success">
<b>What you will gain.</b><br>
You will learn how to construct atom maps from MCS alignments, understand
their failure modes, apply a modern ML-based mapper (RXNMapper), and compare
atom maps rigorously using <b>ITS graph isomorphism</b>.
</div>

<div class="alert alert-block alert-warning">
<b>Prerequisites.</b><br>
You should be comfortable with:
<ul>
  <li>Typed molecular graphs and morphisms (S01)</li>
  <li>Subgraph isomorphism, symmetry, and MCS (S02)</li>
  <li>Basic reaction SMILES notation</li>
</ul>
</div>


## Aim of this talktorial

In **S01**, we formalized molecules as typed graphs and defined
label-preserving morphisms.
In **S02**, we studied subgraph matching and MCS as alignment primitives.

This talktorial (**S03**) brings these ideas into the **reaction domain**.
We view a chemical reaction as two multisets of molecular graphs
(reactants → products) and treat **atom mapping** as a
*label-preserving partial isomorphism* between them.

Concretely, we focus on:

1. **MCS-based atom mapping**  
   Building an atom correspondence by aligning maximum common substructures
   between reactants and products.

2. **Failure modes of naive MCS mapping**  
   Understanding why MCS alone is insufficient:
   symmetry, multi-component ambiguity, and unbalanced reactions.

3. **Attention-guided atom mapping (RXNMapper)**  
   Running RXNMapper to obtain mapped reaction SMILES and extracting atom maps.

4. **ITS graph construction**  
   Encoding a mapped reaction as an **Imaginary Transition State (ITS)** graph.

5. **Map comparison via graph isomorphism**  
   Comparing different atom maps by testing **ITS isomorphism**, making the
   comparison invariant to atom-map IDs.

---

## Learning outcomes

After completing this talktorial, you will be able to:

- Interpret an **atom map** as a label-preserving (partial) graph morphism.
- Construct a simple **MCS-based atom map** in code.
- Identify when and why MCS-based mapping fails.
- Run **RXNMapper** and parse mapped reaction SMILES.
- Build an **ITS graph** from a mapped reaction.
- Decide whether two atom maps are equivalent by checking
  **ITS graph isomorphism**.

---

## Outline

0. **Setup & reaction data**
1. **Atom mapping as a graph morphism**
2. **MCS-based atom mapping (baseline)**
3. **Failure cases: symmetry, components, imbalance**
4. **RXNMapper: attention-based atom mapping**
5. **ITS construction from mapped reactions**
6. **Comparing atom maps via ITS isomorphism**
7. **Discussion, quiz, and references**

> **Connection to earlier notebooks.**  
> - S01 introduced label-preserving morphisms and graph isomorphism.  
> - S02 used these ideas for subgraph matching and MCS.  
> - S03 applies the *same formal machinery* to reactions, showing that
>   atom mapping and ITS comparison are natural graph-theoretic operations.

## 0. Setup & Data

We use:
- **RDKit** for molecules + MCS (`rdFMCS`)
- **NetworkX** for ITS graphs + isomorphism tests
- **rxnmapper** (optional) for a strong baseline mapper

If `rxnmapper` is not installed, the notebook still runs and will skip those parts.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import permutations
from typing import Dict, List, Optional, Tuple

import pandas as pd

import rdkit
from rdkit import Chem
from rdkit.Chem import rdFMCS

import networkx as nx
from networkx.algorithms import isomorphism as iso

print("RDKit:", rdkit.__version__)
print("NetworkX:", nx.__version__)

# Optional: RXNMapper
try:
    from rxnmapper import RXNMapper  # type: ignore
    _HAS_RXNMAPPER = True
    print("rxnmapper: available")
except Exception:
    _HAS_RXNMAPPER = False
    print("rxnmapper: NOT available -> will skip RXNMapper cells")

RDKit: 2025.09.3
NetworkX: 3.6.1


/home/lolo/miniforge3/envs/synedu/lib/python3.11/site-packages/rxnmapper/batched_mapper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/lolo/miniforge3/envs/synedu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rxnmapper: available


## 1. Formal view (bridging from S01 and S02)

### 1.1 Molecules as typed labeled graphs (S01 recap)

In **S01**, a molecule is modeled as a **typed labeled graph**

$$
G = (V, E, a, b),
$$

with the following data:

$$
V \;\text{is a finite set of atoms},
\qquad
E \subseteq \{\{u,v\} \mid u,v \in V,\; u \neq v\},
$$

$$
a : V \longrightarrow \mathcal{A},
\qquad
\mathcal{A} = \text{atom label space},
$$

$$
b : E \longrightarrow \mathcal{B},
\qquad
\mathcal{B} = \text{bond label space}.
$$

A **label-preserving graph morphism**

$$
\varphi : V(G) \longrightarrow V(H)
$$

satisfies:

$$
\text{(M1)} \quad
a_H(\varphi(v)) = a_G(v)
\quad \forall v \in V(G),
$$

$$
\text{(M2)} \quad
\{u,v\} \in E(G)
\;\Rightarrow\;
\{\varphi(u), \varphi(v)\} \in E(H),
$$

$$
\text{(M3)} \quad
b_H(\{\varphi(u), \varphi(v)\})
=
b_G(\{u,v\})
\quad \text{(optional)}.
$$

An **isomorphism** is a bijective morphism whose inverse is also a morphism.

---

### 1.2 Reactions and atom mapping as partial isomorphisms

A reaction SMILES is written as

$$
R \;\gg\; P,
$$

where \(R\) and \(P\) are multisets of molecules.
Each side is represented as a **disjoint union of molecular graphs**:

$$
G_R = \biguplus_i G_{R_i},
\qquad
G_P = \biguplus_j G_{P_j}.
$$

In general,

$$
G_R \not\cong G_P,
$$

because reactions may be unbalanced and some atoms may not participate.

An **atom map** is therefore modeled as a **partial isomorphism**:
find subgraphs

$$
G_R' \subseteq G_R,
\qquad
G_P' \subseteq G_P,
$$

and an isomorphism

$$
\varphi : V(G_R') \longrightarrow V(G_P')
$$

that maximizes

$$
|V(G_R')|.
$$

---

### 1.3 Maximum Common Substructure (MCS) and its role (S02)

As formalized in **S02**, a **Maximum Common Substructure (MCS)** between
\(G_R\) and \(G_P\) is a graph

$$
K = (V_K, E_K, a_K, b_K)
$$

together with injective morphisms

$$
\iota_R : V_K \hookrightarrow V(G_R),
\qquad
\iota_P : V_K \hookrightarrow V(G_P),
$$

such that

$$
K \in
\arg\max_{S}
\left(
|V(S)|
\;\middle|\;
S \hookrightarrow G_R,\;
S \hookrightarrow G_P
\right).
$$

The embeddings \(\iota_R\) and \(\iota_P\) induce an atom map

$$
\varphi
=
\iota_P \circ \iota_R^{-1}
:
V(G_R') \longrightarrow V(G_P'),
$$

where \(G_R' = \iota_R(K)\) and \(G_P' = \iota_P(K)\).

---

### 1.4 From MCS to ITS and map equivalence

Given an atom map \(\varphi\), the **Imaginary Transition State (ITS)** graph
encodes bond changes via paired labels

$$
b_{\mathrm{ITS}} : E_{\mathrm{ITS}} \longrightarrow \mathcal{B}_R \times \mathcal{B}_P.
$$

Two atom maps are considered **equivalent** if their corresponding ITS graphs
are **isomorphic** under label-preserving graph isomorphism, as defined in **S01**.


## 2. Utilities: parsing and RDKit helpers

Conventions:
- mapped atoms are those with `atom.GetAtomMapNum() > 0`
- our teaching MCS mapper allows **partial mapping** (unbalanced OK)

In [ ]:
def split_rxn_smiles(rxn: str) -> Tuple[List[str], List[str]]:
    left, right = rxn.split(">>")
    r_smis = [s for s in left.split(".") if s]
    p_smis = [s for s in right.split(".") if s]
    return r_smis, p_smis


def mol_from_smiles(sm: str) -> Chem.Mol:
    m = Chem.MolFromSmiles(sm)
    if m is None:
        raise ValueError(f"Bad SMILES: {sm}")
    return m


def clear_atom_maps(m: Chem.Mol) -> Chem.Mol:
    mm = Chem.Mol(m)
    for a in mm.GetAtoms():
        a.SetAtomMapNum(0)
    return mm


def apply_atom_maps(m: Chem.Mol, atom_idx_to_map: Dict[int, int]) -> Chem.Mol:
    mm = clear_atom_maps(m)
    for idx, mnum in atom_idx_to_map.items():
        mm.GetAtomWithIdx(int(idx)).SetAtomMapNum(int(mnum))
    return mm


def smiles_with_maps(m: Chem.Mol) -> str:
    # canonical=False preserves input ordering better for didactic diffs
    return Chem.MolToSmiles(m, canonical=False)


def rxn_smiles_with_maps(r_mols: List[Chem.Mol], p_mols: List[Chem.Mol]) -> str:
    left = ".".join(smiles_with_maps(m) for m in r_mols)
    right = ".".join(smiles_with_maps(m) for m in p_mols)
    return f"{left}>>{right}"

## 3. Atom mapping via MCS (teaching version)

### 3.1 Problem with the naive idea

A naive MCS mapper assumes **one reactant vs one product**.
It fails on:
- multi-component reactions (assignment problem),
- symmetry (many equally valid embeddings),
- unbalanced reactions (extra/missing atoms).

### 3.2 Patch for teaching

We extend the mapper to:
- choose a component assignment maximizing total MCS size,
- allow partial mapping (unmatched atoms remain unmapped),
- keep deterministic choices (first embedding) for reproducibility.

In [ ]:
def mcs_atom_map_one(mr: Chem.Mol, mp: Chem.Mol, *, ring_matches_ring_only: bool = True) -> Dict[int, int]:
    # Compute a single atom-index map (mr -> mp) from an MCS pattern.
    # Deterministic: first embedding on each side.
    res = rdFMCS.FindMCS(
        [mr, mp],
        ringMatchesRingOnly=ring_matches_ring_only,
        completeRingsOnly=False,
        matchValences=False,
        matchChiralTag=False,
        bondCompare=rdFMCS.BondCompare.CompareOrder,
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        timeout=10,
    )
    if res.canceled or not res.smartsString:
        return {}

    patt = Chem.MolFromSmarts(res.smartsString)
    if patt is None:
        return {}

    r_matches = mr.GetSubstructMatches(patt, uniquify=False)
    p_matches = mp.GetSubstructMatches(patt, uniquify=False)
    if not r_matches or not p_matches:
        return {}

    rm = r_matches[0]
    pm = p_matches[0]
    return {int(ri): int(pi) for ri, pi in zip(rm, pm)}


@dataclass
class PairMap:
    r_idx: int
    p_idx: int
    atom_map_r_to_p: Dict[int, int]
    mcs_atoms: int


def best_component_assignment(r_mols: List[Chem.Mol], p_mols: List[Chem.Mol]) -> List[PairMap]:
    # Brute-force component assignment (teaching): maximize total mapped atoms.
    nR, nP = len(r_mols), len(p_mols)
    k = min(nR, nP)
    if k == 0:
        return []

    best_score = -1
    best_pairs: List[PairMap] = []

    for p_perm in permutations(range(nP), k):
        total = 0
        pairs: List[PairMap] = []
        ok = True
        for r_idx, p_idx in enumerate(p_perm):
            amap = mcs_atom_map_one(r_mols[r_idx], p_mols[p_idx])
            if not amap:
                ok = False
                break
            total += len(amap)
            pairs.append(PairMap(r_idx=r_idx, p_idx=p_idx, atom_map_r_to_p=amap, mcs_atoms=len(amap)))
        if ok and total > best_score:
            best_score = total
            best_pairs = pairs

    return best_pairs


def make_mcs_mapped_rxn(rxn: str) -> Optional[str]:
    # Return mapped reaction SMILES using MCS-derived partial mapping.
    # Unmatched atoms remain unmapped.
    r_smis, p_smis = split_rxn_smiles(rxn)
    r_mols = [mol_from_smiles(s) for s in r_smis]
    p_mols = [mol_from_smiles(s) for s in p_smis]

    pairs = best_component_assignment(r_mols, p_mols)
    if not pairs:
        return None

    next_map = 1
    r_maps: List[Dict[int, int]] = [dict() for _ in r_mols]
    p_maps: List[Dict[int, int]] = [dict() for _ in p_mols]

    for pair in pairs:
        for r_ai, p_ai in pair.atom_map_r_to_p.items():
            mnum = next_map
            next_map += 1
            r_maps[pair.r_idx][r_ai] = mnum
            p_maps[pair.p_idx][p_ai] = mnum

    r_out = [apply_atom_maps(m, mp) for m, mp in zip(r_mols, r_maps)]
    p_out = [apply_atom_maps(m, mp) for m, mp in zip(p_mols, p_maps)]
    return rxn_smiles_with_maps(r_out, p_out)

In [ ]:
def mcs_atom_map_one(
    mr: Chem.Mol,
    mp: Chem.Mol,
    *,
    relax: bool = False,
    ring_matches_ring_only: bool = True,
) -> Dict[int, int]:
    """
    Compute a single atom-index map (mr -> mp) from an MCS pattern.

    relax=False:
        strict morphism (preserve bond order, valence, rings)
    relax=True:
        relaxed morphism (ignore valence, allow bond-order mismatch)
    """

    if relax:
        bond_compare = rdFMCS.BondCompare.CompareAny
        match_valences = False
        complete_rings = False
    else:
        bond_compare = rdFMCS.BondCompare.CompareOrder
        match_valences = True
        complete_rings = True

    res = rdFMCS.FindMCS(
        [mr, mp],
        ringMatchesRingOnly=ring_matches_ring_only,
        completeRingsOnly=complete_rings,
        matchValences=match_valences,
        matchChiralTag=not relax,
        bondCompare=bond_compare,
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        timeout=10,
    )

    if res.canceled or not res.smartsString:
        return {}

    patt = Chem.MolFromSmarts(res.smartsString)
    if patt is None:
        return {}

    r_matches = mr.GetSubstructMatches(patt, uniquify=False)
    p_matches = mp.GetSubstructMatches(patt, uniquify=False)
    if not r_matches or not p_matches:
        return {}

    # deterministic choice
    rm = r_matches[0]
    pm = p_matches[0]
    return {int(ri): int(pi) for ri, pi in zip(rm, pm)}

def best_component_assignment(
    r_mols: List[Chem.Mol],
    p_mols: List[Chem.Mol],
    *,
    relax: bool = False,
) -> List[PairMap]:
    nR, nP = len(r_mols), len(p_mols)
    k = min(nR, nP)
    if k == 0:
        return []

    best_score = -1
    best_pairs: List[PairMap] = []

    for p_perm in permutations(range(nP), k):
        total = 0
        pairs: List[PairMap] = []
        ok = True
        for r_idx, p_idx in enumerate(p_perm):
            amap = mcs_atom_map_one(
                r_mols[r_idx],
                p_mols[p_idx],
                relax=relax,
            )
            if not amap:
                ok = False
                break
            total += len(amap)
            pairs.append(
                PairMap(
                    r_idx=r_idx,
                    p_idx=p_idx,
                    atom_map_r_to_p=amap,
                    mcs_atoms=len(amap),
                )
            )
        if ok and total > best_score:
            best_score = total
            best_pairs = pairs

    return best_pairs

def make_mcs_mapped_rxn(rxn: str, *, relax: bool = False) -> Optional[str]:
    r_smis, p_smis = split_rxn_smiles(rxn)
    r_mols = [mol_from_smiles(s) for s in r_smis]
    p_mols = [mol_from_smiles(s) for s in p_smis]

    pairs = best_component_assignment(r_mols, p_mols, relax=relax)
    if not pairs:
        return None

    next_map = 1
    r_maps: List[Dict[int, int]] = [dict() for _ in r_mols]
    p_maps: List[Dict[int, int]] = [dict() for _ in p_mols]

    for pair in pairs:
        for r_ai, p_ai in pair.atom_map_r_to_p.items():
            mnum = next_map
            next_map += 1
            r_maps[pair.r_idx][r_ai] = mnum
            p_maps[pair.p_idx][p_ai] = mnum

    r_out = [apply_atom_maps(m, mp) for m, mp in zip(r_mols, r_maps)]
    p_out = [apply_atom_maps(m, mp) for m, mp in zip(p_mols, p_maps)]
    return rxn_smiles_with_maps(r_out, p_out)


### 3.3 Demo set (balanced, multi-component, symmetry, unbalanced)

In [ ]:
demo_rxns = [
    ("ethanol_to_acetaldehyde", "CCO>>CC=O"),
    ("sn2_substitution", "CCCl.[OH-]>>CCO.[Cl-]"),
    ("benzene_chlorination", "c1ccccc1.Cl>>c1ccccc1Cl"),
    ("unbalanced_demo", "CCO>>CC=O.O"),
]

for name, rxn in demo_rxns:
    mapped = make_mcs_mapped_rxn(rxn, relax=True)
    print(f"{name:>24} | {rxn:>28} | MCS mapped? {mapped is not None}")
    if mapped:
        print("  ", mapped)

### 3.4 Symmetry deep dive: counting MCS embeddings

In symmetric systems, MCS has many matches (S01: automorphisms).
We count how many embeddings RDKit finds for benzene → chlorobenzene ring conservation.

In [ ]:
def count_mcs_embeddings(sm_r: str, sm_p: str) -> Tuple[int, int, int]:
    mr = mol_from_smiles(sm_r)
    mp = mol_from_smiles(sm_p)

    res = rdFMCS.FindMCS(
        [mr, mp],
        ringMatchesRingOnly=True,
        bondCompare=rdFMCS.BondCompare.CompareOrder,
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        timeout=10,
    )
    patt = Chem.MolFromSmarts(res.smartsString) if res.smartsString else None
    if patt is None:
        return (0, 0, 0)

    r_matches = mr.GetSubstructMatches(patt, uniquify=False)
    p_matches = mp.GetSubstructMatches(patt, uniquify=False)
    return (patt.GetNumAtoms(), len(r_matches), len(p_matches))


mcs_atoms, r_emb, p_emb = count_mcs_embeddings("c1ccccc1", "c1ccccc1Cl")
print("MCS atoms:", mcs_atoms)
print("reactant embeddings:", r_emb)
print("product embeddings:", p_emb)
print("=> many equally valid ring alignments (symmetry).")

## 4. RXNMapper (attention-guided atom mapping)

RXNMapper outputs:
- `mapped_rxn`: reaction SMILES with atom-map numbers
- `confidence`: a float score

If unavailable, the notebook skips these cells.

In [ ]:
def rxnmapper_map(rxn: str) -> Tuple[Optional[str], Optional[float]]:
    if not _HAS_RXNMAPPER:
        return None, None
    mapper = RXNMapper()
    out = mapper.get_attention_guided_atom_maps([rxn])
    mapped = out[0].get("mapped_rxn", None)
    conf = out[0].get("confidence", None)
    return mapped, conf


if _HAS_RXNMAPPER:
    for name, rxn in demo_rxns:
        mapped, conf = rxnmapper_map(rxn)
        print(f"{name:>24} | conf={conf:.3f} | {mapped}")
else:
    print("rxnmapper not available; install with: pip install rxnmapper")

## 5. ITS graph construction

ITS nodes are map numbers. ITS edges carry labels `(br, bp)`:
- `br`: bond order in reactants (0 if absent)
- `bp`: bond order in products (0 if absent)

Reaction center edges are those with `br != bp`.

In [ ]:
def bond_order(b: Chem.Bond) -> int:
    bt = b.GetBondType()
    if bt == Chem.BondType.SINGLE:
        return 1
    if bt == Chem.BondType.DOUBLE:
        return 2
    if bt == Chem.BondType.TRIPLE:
        return 3
    if bt == Chem.BondType.AROMATIC:
        return 1  # teaching choice
    return 0


def mols_from_side(sm_side: str) -> List[Chem.Mol]:
    return [mol_from_smiles(s) for s in sm_side.split(".") if s]


def add_mapped_atom_nodes(G: nx.Graph, side: str, mols: List[Chem.Mol]) -> None:
    for m in mols:
        for a in m.GetAtoms():
            mp = a.GetAtomMapNum()
            if not mp:
                continue
            mp = int(mp)
            if mp not in G:
                G.add_node(mp)
            G.nodes[mp][f"{side}_symbol"] = a.GetSymbol()
            G.nodes[mp][f"{side}_charge"] = int(a.GetFormalCharge())
            G.nodes[mp][f"{side}_arom"] = bool(a.GetIsAromatic())


def side_bond_order(mols: List[Chem.Mol], mpa: int, mpb: int) -> int:
    for m in mols:
        idx_a = idx_b = None
        for a in m.GetAtoms():
            if a.GetAtomMapNum() == mpa:
                idx_a = a.GetIdx()
            elif a.GetAtomMapNum() == mpb:
                idx_b = a.GetIdx()
        if idx_a is None or idx_b is None:
            continue
        b = m.GetBondBetweenAtoms(int(idx_a), int(idx_b))
        if b is None:
            return 0
        return bond_order(b)
    return 0


def build_its_from_mapped_rxn(mapped_rxn: str) -> nx.Graph:
    r_side, p_side = mapped_rxn.split(">>")
    r_mols = mols_from_side(r_side)
    p_mols = mols_from_side(p_side)

    G = nx.Graph()
    add_mapped_atom_nodes(G, "R", r_mols)
    add_mapped_atom_nodes(G, "P", p_mols)

    maps = list(G.nodes())
    for i in range(len(maps)):
        for j in range(i + 1, len(maps)):
            a, b = maps[i], maps[j]
            br = side_bond_order(r_mols, a, b)
            bp = side_bond_order(p_mols, a, b)
            if br != 0 or bp != 0:
                G.add_edge(a, b, br=int(br), bp=int(bp))

    return G


def reaction_center_edges(Gits: nx.Graph) -> List[Tuple[int, int]]:
    rc = []
    for u, v, d in Gits.edges(data=True):
        if int(d.get("br", 0)) != int(d.get("bp", 0)):
            rc.append((u, v))
    return rc


mapped = make_mcs_mapped_rxn("CCO>>CC=O", relax=True)
G = build_its_from_mapped_rxn(mapped)
print("ITS nodes:", G.number_of_nodes(), "ITS edges:", G.number_of_edges())
print("reaction-center edges:", reaction_center_edges(G))

## 6. Comparing atom maps via ITS isomorphism (map-number invariant)

We use `networkx.GraphMatcher` (S01) with:
- `node_match`: compare node labels (`R_*`, `P_*`)
- `edge_match`: compare `(br,bp)`

In [ ]:
import re
def its_isomorphic(G1: nx.Graph, G2: nx.Graph) -> bool:
    keys = ["R_symbol", "R_charge", "R_arom", "P_symbol", "P_charge", "P_arom"]

    def node_match(n1, n2) -> bool:
        for k in keys:
            if n1.get(k, None) != n2.get(k, None):
                return False
        return True

    def edge_match(e1, e2) -> bool:
        return (int(e1.get("br", 0)), int(e1.get("bp", 0))) == (int(e2.get("br", 0)), int(e2.get("bp", 0)))

    GM = iso.GraphMatcher(G1, G2, node_match=node_match, edge_match=edge_match)
    return GM.is_isomorphic()


def permute_map_numbers(mapped_rxn: str, shift: int = 100) -> str:
    def repl(m):
        return f":{int(m.group(1)) + shift}"
    return re.sub(r":(\d+)", repl, mapped_rxn)


m1 = make_mcs_mapped_rxn("CCO>>CC=O", relax=True)
m2 = permute_map_numbers(m1, shift=50)

G1 = build_its_from_mapped_rxn(m1)
G2 = build_its_from_mapped_rxn(m2)

print("mapped1:", m1)
print("mapped2:", m2)
print("ITS isomorphic?", its_isomorphic(G1, G2))

## 7. End-to-end table: MCS mapper vs RXNMapper

We compute:
- `mcs_mapped`
- `rxnmapper_mapped` + `conf` (if available)
- `ITS_iso(MCS,RXNMapper)` when both exist

In [ ]:
rows = []
for name, rxn in demo_rxns:
    mcs_mapped = make_mcs_mapped_rxn(rxn, relax=True)
    rxnmapped, conf = rxnmapper_map(rxn) if _HAS_RXNMAPPER else (None, None)

    its_iso_val = None
    if mcs_mapped and rxnmapped:
        try:
            its_iso_val = its_isomorphic(
                build_its_from_mapped_rxn(mcs_mapped),
                build_its_from_mapped_rxn(rxnmapped),
            )
        except Exception:
            its_iso_val = None

    rows.append(
        {
            "name": name,
            "rxn": rxn,
            "mcs_mapped": mcs_mapped,
            "rxnmapper_mapped": rxnmapped,
            "conf": conf,
            "ITS_iso(MCS,RXNMapper)": its_iso_val,
        }
    )

df = pd.DataFrame(rows)
df

## 8. Bonus: reaction center from ITS

Reaction center edges:
\[
RC = \{ (u,v)\in E(G_{ITS}) : b_R(u,v)\neq b_P(u,v)\}.
\]

This is a lightweight extractor that doesn't need templates.

In [ ]:
def reaction_center_summary(mapped_rxn: str) -> pd.DataFrame:
    G = build_its_from_mapped_rxn(mapped_rxn)
    rc = reaction_center_edges(G)
    data = []
    for u, v in rc:
        d = G.edges[u, v]
        data.append({"u": u, "v": v, "br": d.get("br", 0), "bp": d.get("bp", 0)})
    return pd.DataFrame(data)


print("MCS-mapped example:", make_mcs_mapped_rxn("CCO>>CC=O"))
reaction_center_summary(make_mcs_mapped_rxn("CCO>>CC=O"))

## 9. Exercises

### Exercise 1 (S01 link): make the isomorphism “too permissive”
Modify `node_match` in `its_isomorphic` to ignore charge and compare results on `sn2_substitution`.

### Exercise 2: symmetry
Force a different MCS embedding for benzene chlorination (swap which match you pick) and show ITS-isomorphism stays `True`.

### Exercise 3: scaling
Why does brute-force component assignment explode? Estimate complexity for `n` components.

### Exercise 4: aromatic handling
Encode aromatic bonds with a special integer (e.g., 15) and see if comparisons change.

## 10. Takeaways

- Atom mapping can be formalized as a **partial isomorphism** between reaction-side graphs (S01 morphisms).
- MCS is a principled baseline but needs help for multi-component, symmetry, and unbalanced cases.
- RXNMapper provides a strong practical mapping + confidence.
- ITS graphs give a compact transformation representation.
- ITS isomorphism is a clean map-number-invariant way to compare atom maps.